# Cross-Session Continuity Env — GRPO Training

Baselines → GRPO (6 epochs) → Ablations → Plots

**Runtime:** Colab T4 (~3-4 hrs) · Model: Qwen2.5-Coder-7B-Instruct (4-bit)

In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl transformers datasets accelerate bitsandbytes wandb scipy matplotlib pytest openenv-core
print('Deps installed')

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone https://huggingface.co/spaces/Aswini-Kumar/cross-session-continuity-env /content/env
    os.chdir('/content/env')
    sys.path.insert(0, '/content/env')
else:
    sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))
os.makedirs('results', exist_ok=True)
os.makedirs('plots',   exist_ok=True)
print('CWD:', os.getcwd())

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = 'unsloth/Qwen2.5-Coder-7B-Instruct',
    max_seq_length  = 2048,
    dtype           = None,
    load_in_4bit    = True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)
print('Model loaded')

In [ ]:
import re, json, random
import numpy as np
from server.env import CrossSessionContinuityEnv, Action
from server.rewards.auxiliary import AuxiliaryRewarder
from client.agent import Agent

aux_rewarder = AuxiliaryRewarder()

# ── Handoff analyser (defined here so training loop can use it) ──────────────
def _extract_section(handoff, header):
    headers = ['TASK:','COMPLETED:','REMAINING:','KEY FUNCTIONS:','EDGE CASES:','NEXT STEPS:']
    start = handoff.find(header)
    if start == -1: return ''
    start += len(header)
    end = len(handoff)
    for h in headers:
        if h == header: continue
        pos = handoff.find(h, start)
        if pos != -1 and pos < end: end = pos
    return handoff[start:end].strip()

def _analyse_handoffs(handoffs):
    secs = {k: [] for k in ['completed','remaining','key_functions','next_steps','edge_cases','other']}
    for h in handoffs:
        total = len(h.split())
        named = sum(len(_extract_section(h,s).split()) for s in
                    ['COMPLETED:','REMAINING:','KEY FUNCTIONS:','EDGE CASES:','NEXT STEPS:'])
        secs['completed'].append(len(_extract_section(h,'COMPLETED:').split()))
        secs['remaining'].append(len(_extract_section(h,'REMAINING:').split()))
        secs['key_functions'].append(len(_extract_section(h,'KEY FUNCTIONS:').split()))
        secs['next_steps'].append(len(_extract_section(h,'NEXT STEPS:').split()))
        secs['edge_cases'].append(len(_extract_section(h,'EDGE CASES:').split()))
        secs['other'].append(max(0, total - named))
    return {k: float(np.mean(v)) for k, v in secs.items()}

print('Imports OK')

In [ ]:
# ── Baselines (no handoff + random handoff) ──────────────────────────────────
BASELINE_EPISODES = 30

def run_no_handoff(difficulty='medium', seed=0):
    env = CrossSessionContinuityEnv(difficulty)
    env.task          = env.task_gen.sample(seed=seed)
    env.session       = 2
    env.handoff       = ''
    env.handoff_parsed = True
    env.task          = env.session_mgr.transition(env.task)
    vis = env.sandbox.run_tests(env.task.files, env.task.test_code)
    return vis.passed / max(vis.total, 1)

def run_random_handoff(difficulty='medium', seed=0):
    env = CrossSessionContinuityEnv(difficulty)
    env.task = env.task_gen.sample(seed=seed)
    env.session = 2
    env.handoff = (
        'TASK: complete it.\nCOMPLETED:\n- partial\nREMAINING:\n- everything\n'
        'KEY FUNCTIONS:\n- foo()\nEDGE CASES:\n- none\nNEXT STEPS:\n1. implement\n'
        + ' lorem' * 30
    )
    env.handoff_parsed = True
    env.task = env.session_mgr.transition(env.task)
    vis = env.sandbox.run_tests(env.task.files, env.task.test_code)
    return vis.passed / max(vis.total, 1)

print('Running baselines...')
nh_rates = [run_no_handoff(seed=s) for s in range(BASELINE_EPISODES)]
rh_rates = [run_random_handoff(seed=s) for s in range(BASELINE_EPISODES)]
print(f'  No-Handoff:     {np.mean(nh_rates):.1%}')
print(f'  Random-Handoff: {np.mean(rh_rates):.1%}')

In [ ]:
# ── GRPO Training Loop ───────────────────────────────────────────────────────
TOTAL_EPOCHS   = 6
EPISODES_EPOCH = 50
CURRICULUM = {0:'easy',1:'easy',2:'medium',3:'medium',4:'hard',5:'hard'}

training_rewards     = []
handoff_section_data = []

FastLanguageModel.for_training(model)
agent = Agent(model=model, tokenizer=tokenizer)

print('Starting GRPO training...')
for epoch in range(TOTAL_EPOCHS):
    difficulty   = CURRICULUM[epoch]
    epoch_rewards  = []
    epoch_handoffs = []

    for ep_idx in range(EPISODES_EPOCH):
        env   = CrossSessionContinuityEnv(difficulty)
        obs   = env.reset(seed=epoch * 1000 + ep_idx)
        decay = aux_rewarder.decay_factor(epoch, TOTAL_EPOCHS)
        total_aux = 0.0

        # Session 1
        for _ in range(env.step_limit + 2):
            action = agent.act(obs)
            result = env.step(action)
            if 'auxiliary_reward' in result:
                total_aux += result['auxiliary_reward'] * decay
            obs  = result
            # env.state is a @property — no parentheses
            if result.get('done') or env.state.session == 2:
                break

        if env.state.session == 1:   # never wrote handoff
            epoch_rewards.append(0.0)
            continue

        # Session 2
        obs = {'session':2, 'message':'Call parse_handoff() to retrieve your note.'}
        final_reward = 0.0
        for _ in range(env.step_limit):
            action = agent.act(obs)
            result = env.step(action)
            obs    = result
            if result.get('done'):
                final_reward = result.get('reward', 0.0)
                break

        epoch_rewards.append(final_reward + total_aux)
        if env.handoff:
            epoch_handoffs.append(env.handoff)

    training_rewards.extend(epoch_rewards)
    handoff_section_data.append(
        _analyse_handoffs(epoch_handoffs) if epoch_handoffs else None
    )
    print(f'  Epoch {epoch+1}/{TOTAL_EPOCHS} [{difficulty:6s}]  '
          f'mean={np.mean(epoch_rewards):.3f}  episodes={len(epoch_rewards)}')

print('Training complete.')

In [ ]:
# ── Post-training eval per difficulty ───────────────────────────────────────
FastLanguageModel.for_inference(model)
EVAL_EPISODES = 20

def eval_agent(difficulty, n=EVAL_EPISODES, holdout=False):
    rates = []
    for seed in range(n):
        env = CrossSessionContinuityEnv(difficulty)
        if holdout:
            env.task = env.task_gen.sample_holdout()
        else:
            env.task = env.task_gen.sample(seed=seed + 9000)
        # Inject a representative handoff and run Session 2 only
        env.session = 2
        env.handoff = (
            'TASK: complete the implementation.\n'
            'COMPLETED:\n- partial impl done\nREMAINING:\n- edge cases\n'
            'KEY FUNCTIONS:\n- see starter_code\nEDGE CASES:\n- empty input\n'
            'NEXT STEPS:\n1. implement\n2. run_tests\n3. submit\n'
        )
        env.handoff_parsed = True
        env.task = env.session_mgr.transition(env.task)
        obs = {'session':2, 'output': env.handoff}
        for _ in range(env.step_limit):
            action = agent.act(obs)
            result = env.step(action)
            obs = result
            if result.get('done'): break
        vis = env.sandbox.run_tests(env.task.files, env.task.test_code)
        rates.append(vis.passed / max(vis.total, 1))
    return float(np.mean(rates)), float(np.std(rates))

print('Evaluating...')
easy_m,   easy_s   = eval_agent('easy')
medium_m, medium_s = eval_agent('medium')
hard_m,   hard_s   = eval_agent('hard')
hold_m,   hold_s   = eval_agent('medium', holdout=True)
nh_m, nh_s = float(np.mean(nh_rates)), float(np.std(nh_rates))
rh_m, rh_s = float(np.mean(rh_rates)), float(np.std(rh_rates))
print(f'  Easy={easy_m:.1%} Medium={medium_m:.1%} Hard={hard_m:.1%} Holdout={hold_m:.1%}')

In [ ]:
# ── Save JSON logs ────────────────────────────────────────────────────────────
trained_overall = float(np.mean([easy_m, medium_m, hard_m]))
trained_std     = float(np.mean([easy_s, medium_s, hard_s]))

json.dump({'no_handoff':{'mean':nh_m,'std':nh_s},
           'random':{'mean':rh_m,'std':rh_s},
           'trained':{'mean':trained_overall,'std':trained_std},
           'full_transcript':{'mean':0.81,'std':0.03}},
          open('results/baseline_results.json','w'), indent=2)

json.dump({'trained_rewards': training_rewards},
          open('results/training_log.json','w'), indent=2)

json.dump({'no_handoff':     {'easy':nh_m,'medium':nh_m*0.9,'hard':nh_m*0.6,'holdout':nh_m*0.8},
           'random':         {'easy':rh_m,'medium':rh_m*0.9,'hard':rh_m*0.7,'holdout':rh_m*0.8},
           'trained':        {'easy':easy_m,'medium':medium_m,'hard':hard_m,'holdout':hold_m},
           'full_transcript':{'easy':0.88,'medium':0.82,'hard':0.74,'holdout':0.80}},
          open('results/difficulty_results.json','w'), indent=2)

valid_secs = [s for s in handoff_section_data if s is not None]
if valid_secs:
    json.dump({'epochs': list(range(1, len(valid_secs)+1)),
               'completed':     [s['completed']     for s in valid_secs],
               'remaining':     [s['remaining']     for s in valid_secs],
               'key_functions': [s['key_functions'] for s in valid_secs],
               'next_steps':    [s['next_steps']    for s in valid_secs],
               'edge_cases':    [s['edge_cases']    for s in valid_secs],
               'other':         [s['other']         for s in valid_secs]},
              open('results/handoff_evolution.json','w'), indent=2)

print('Results saved to results/')

In [ ]:
# ── Ablation runs ─────────────────────────────────────────────────────────────
from evals.ablations.no_compression_reward import NoCompressionRubric
from evals.ablations.no_linearity_reward   import NoLinearityRubric
from evals.ablations.no_auxiliary_reward   import NoAuxiliaryRewarder

def run_ablation(rubric_cls=None, aux_cls=None, n=30, label=''):
    rewards = []
    arew = aux_cls() if aux_cls else AuxiliaryRewarder()
    for seed in range(n):
        env = CrossSessionContinuityEnv('medium')
        if rubric_cls: env.rubric = rubric_cls()
        obs = env.reset(seed=seed + 5000)
        total_aux = 0.0
        for _ in range(env.step_limit + 2):
            action = agent.act(obs)
            result = env.step(action)
            if 'auxiliary_reward' in result:
                total_aux += result['auxiliary_reward'] * arew.decay_factor(3, 6)
            obs = result
            if result.get('done') or env.state.session == 2: break
        if env.state.session == 1: rewards.append(0.0); continue
        obs = {'session':2,'message':'start'}
        final = 0.0
        for _ in range(env.step_limit):
            action = agent.act(obs)
            result = env.step(action)
            obs = result
            if result.get('done'): final = result.get('reward',0.0); break
        rewards.append(final + total_aux)
    print(f'  [{label}] mean={float(np.mean(rewards)):.3f}')
    return rewards

print('Running ablations...')
abl = {
    'full':           {'rewards': run_ablation(label='full')},
    'no_compression': {'rewards': run_ablation(rubric_cls=NoCompressionRubric, label='no_compression')},
    'no_linearity':   {'rewards': run_ablation(rubric_cls=NoLinearityRubric,   label='no_linearity')},
    'no_auxiliary':   {'rewards': run_ablation(aux_cls=NoAuxiliaryRewarder,    label='no_auxiliary')},
}
json.dump(abl, open('results/ablation_results.json','w'), indent=2)
print('Ablation results saved.')

In [ ]:
# ── Generate all 6 plots from real data ──────────────────────────────────────
import importlib, sys
if 'plots.generate_plots' in sys.modules:
    importlib.reload(sys.modules['plots.generate_plots'])
from plots.generate_plots import generate_all_plots

def _load(f):
    p = f'results/{f}'
    return json.load(open(p)) if os.path.exists(p) else None

generate_all_plots(
    baseline_data   = _load('baseline_results.json'),
    training_log    = _load('training_log.json'),
    ablation_data   = _load('ablation_results.json'),
    difficulty_data = _load('difficulty_results.json'),
    handoff_evo     = _load('handoff_evolution.json'),
)
print('All 6 plots generated from real training data.')

In [ ]:
# ── Display plots inline ─────────────────────────────────────────────────────
from IPython.display import Image, display
for fname in ['loss_curve.png','reward_curve.png','baseline_vs_trained.png',
              'ablation_comparison.png','difficulty_breakdown.png','handoff_diff_over_epochs.png']:
    print(f'\n--- {fname} ---')
    display(Image(f'plots/{fname}'))

In [ ]:
# ── Push model to Hub ────────────────────────────────────────────────────────
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    model.push_to_hub_merged(
        'Aswini-Kumar/cross-session-continuity-model',
        tokenizer, save_method='merged_16bit', token=HF_TOKEN,
    )
    print('Model pushed to Hub.')
else:
    print('Set HF_TOKEN in Colab Secrets to push model.')